In [ ]:
# 1. Configuracion

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "si7006_t2"
TBL_SILVER   = f"{CATALOG}.{SCHEMA}.orders_silver"
TBL_GOLD_KPI = f"{CATALOG}.{SCHEMA}.gold_kpi_ventana"

WINDOW_SIZE = "1 minute"

print(f"Silver:    {TBL_SILVER}")
print(f"Gold KPI:  {TBL_GOLD_KPI}")
print(f"Ventana:   {WINDOW_SIZE}")


In [ ]:
# 2. Lectura batch desde Silver

df_silver = spark.table(TBL_SILVER)
df_silver.printSchema()
print(f"Filas Silver: {df_silver.count():,}")


In [ ]:
# 3. Construccion de KPIs por ventana

df_gold_kpi = (
    df_silver
        .groupBy(F.window("event_time", WINDOW_SIZE).alias("w"))
        .agg(
            F.count("*").alias("line_items"),
            F.countDistinct("invoice_no").alias("orders_count"),
            F.sum("quantity").alias("units_sold"),
            F.round(F.sum("total_amount"), 2).alias("gross_revenue"),
            F.round(F.avg("total_amount"), 2).alias("avg_line_amount"),
            F.countDistinct("stock_code").alias("distinct_products"),
            F.countDistinct("country").alias("distinct_countries"),
            F.countDistinct(F.when(F.col("is_guest"), F.col("invoice_no"))).alias("guest_orders")
        )
        .withColumn(
            "avg_order_value",
            F.round(F.col("gross_revenue") / F.col("orders_count"), 2)
        )
        .select(
            F.col("w.start").alias("window_start"),
            F.col("w.end").alias("window_end"),
            "line_items",
            "orders_count",
            "units_sold",
            "gross_revenue",
            "avg_line_amount",
            "avg_order_value",
            "distinct_products",
            "distinct_countries",
            "guest_orders"
        )
        .orderBy("window_start")
)

df_gold_kpi.show(20, truncate=False)


In [ ]:
# 4. Escritura idempotente en Delta

(
    df_gold_kpi.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TBL_GOLD_KPI)
)

print("Tabla gold_kpi_ventana actualizada correctamente")


In [ ]:
# 5. Validacion rapida

spark.table(TBL_GOLD_KPI).printSchema()

spark.sql(f"""
    SELECT COUNT(*) AS ventanas,
           MIN(window_start) AS primer_inicio,
           MAX(window_end)   AS ultimo_fin,
           ROUND(SUM(gross_revenue), 2) AS revenue_total,
           SUM(orders_count) AS pedidos_total
    FROM {TBL_GOLD_KPI}
""").show(truncate=False)

spark.sql(f"SELECT * FROM {TBL_GOLD_KPI} ORDER BY window_start DESC LIMIT 10").show(truncate=False)
